In [1]:
import os

In [2]:
%pwd

'd:\\Revision\\Covid-19_detection_with_MLOPS\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Revision\\Covid-19_detection_with_MLOPS'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    final_dataset_dir: Path

In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir),
            final_dataset_dir=Path(config.final_dataset_dir)
        )

        return data_ingestion_config

     

In [9]:
import os
import shutil
import zipfile
from pathlib import Path
from dotenv import load_dotenv
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size


# Load environment variables from the .env file BEFORE importing kaggle
load_dotenv()
import kaggle

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        """
        Downloads the dataset using the Kaggle API securely via .env credentials.
        """
        if not os.path.exists(self.config.local_data_file):
            logger.info("Authenticating with Kaggle API...")
            
            # Parse the dataset slug from the URL
            # E.g., 'https://.../datasets/tawsifurrahman/covid19-radiography-database' 
            # becomes 'tawsifurrahman/covid19-radiography-database'
            url_parts = self.config.source_URL.split('/')
            dataset_slug = f"{url_parts[-2]}/{url_parts[-1]}"
            
            logger.info(f"Downloading Kaggle dataset: {dataset_slug}")
            
            # Authenticate using the environment variables
            kaggle.api.authenticate()
            
            # Download the zip file to the root directory
            kaggle.api.dataset_download_files(
                dataset_slug, 
                path=self.config.root_dir, 
                unzip=False
            )
            
            # Kaggle saves the file using the dataset name (e.g., covid19-radiography-database.zip)
            # We rename it to our standard 'data.zip' so the rest of the pipeline works seamlessly
            downloaded_zip_name = f"{url_parts[-1]}.zip"
            downloaded_zip_path = os.path.join(self.config.root_dir, downloaded_zip_name)
            
            if os.path.exists(downloaded_zip_path):
                os.rename(downloaded_zip_path, self.config.local_data_file)
            
            logger.info(f"Dataset downloaded successfully to {self.config.local_data_file}")
        else:
            logger.info(f"File already exists. Size: {get_size(Path(self.config.local_data_file))}")  

    def extract_zip_file(self):
        """
        Extracts the raw zip archive into a temporary extraction directory.
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Extracted zip file into: {unzip_path}")

    def clean_and_restructure_dataset(self):
        """
        Parses the Kaggle structure, filtering out masks and metadata.
        Moves only valid X-ray images into standard class directories.
        """
        logger.info("Restructuring dataset to standard format...")
        target_classes = ["COVID", "Normal"]
        
        raw_root = None
        for path in Path(self.config.unzip_dir).rglob("COVID"):
            if path.is_dir():
                raw_root = path.parent
                break
                
        if not raw_root:
            raise FileNotFoundError("Could not find the extracted dataset structure. Check zip contents.")

        for cls in target_classes:
            dest_dir = Path(self.config.final_dataset_dir) / cls
            os.makedirs(dest_dir, exist_ok=True)
            
            source_img_dir = raw_root / cls / "images"
            
            if source_img_dir.exists():
                logger.info(f"Moving images from {source_img_dir} to {dest_dir}...")
                for img_file in source_img_dir.iterdir():
                    if img_file.is_file() and img_file.suffix.lower() in ['.png', '.jpg', '.jpeg']:
                        shutil.move(str(img_file), str(dest_dir / img_file.name))
            else:
                logger.warning(f"Expected source folder {source_img_dir} not found.")

        # Cleanup raw files
        if os.path.exists(self.config.unzip_dir):
            shutil.rmtree(self.config.unzip_dir)
            logger.info("Cleaned up raw temporary extraction files.")

In [11]:


STAGE_NAME = "Data Ingestion Stage"

try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
    data_ingestion.clean_and_restructure_dataset()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e)
    raise e

[2026-06-27 00:02:28,193: INFO: 1356313550: >>>>>> stage Data Ingestion Stage started <<<<<<]
[2026-06-27 00:02:28,194: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-27 00:02:28,196: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-27 00:02:28,204: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-27 00:02:28,205: INFO: common: created directory at: artifacts]
[2026-06-27 00:02:28,206: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-27 00:02:28,207: INFO: 605447640: Authenticating with Kaggle API...]
[2026-06-27 00:02:28,208: INFO: 605447640: Downloading Kaggle dataset: tawsifurrahman/covid19-radiography-database]
Dataset URL: https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database
[2026-06-27 00:06:14,502: INFO: 605447640: Dataset downloaded successfully to artifacts\data_ingestion\data.zip]
[2026-06-27 00:06:28,619: INFO: 605447640: Extracted zip file into: artifacts\data_ingesti